In [1]:
!pip install transformers datasets accelerate -q

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import DataCollatorWithPadding
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import torch

In [3]:
df = pd.read_csv("C:/Users/pabit/Downloads/final_dataset.csv")
print("After exact dedup:", len(df))

After exact dedup: 63074


In [4]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])

print(len(train_df), len(val_df), len(test_df))

44151 9461 9462


In [5]:
train_set = set(train_df["combined"])

print("Train-Val overlap:",
      val_df["combined"].isin(train_set).sum())

print("Train-Test overlap:",
      test_df["combined"].isin(train_set).sum())

Train-Val overlap: 0
Train-Test overlap: 0


In [6]:
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["combined"],
        truncation=True,
        max_length=512
    )

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

train_ds = Dataset.from_pandas(train_df[["combined", "label"]].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[["combined", "label"]].reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df[["combined", "label"]].reset_index(drop=True))

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds   = val_ds.rename_column("label", "labels")
test_ds  = test_ds.rename_column("label", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/44151 [00:00<?, ? examples/s]

Map:   0%|          | 0/9461 [00:00<?, ? examples/s]

Map:   0%|          | 0/9462 [00:00<?, ? examples/s]

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
import math


batch_size = 16
epochs = 3
warmup_ratio = 0.1

steps_per_epoch = math.ceil(len(train_ds) / batch_size)
total_steps = steps_per_epoch * epochs
warmup_steps = int(total_steps * warmup_ratio)


args = TrainingArguments(
    output_dir="./roberta-fakenews",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,           
    logging_steps=100,
    report_to="none"
)

In [10]:
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.011323,0.010590,0.997780,0.997760
2,0.004020,0.010326,0.998626,0.998614
3,0.000012,0.007843,0.998943,0.998934


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8280, training_loss=0.022385128191559786, metrics={'train_runtime': 2875.3513, 'train_samples_per_second': 46.065, 'train_steps_per_second': 2.88, 'total_flos': 3.484984861559808e+16, 'train_loss': 0.022385128191559786, 'epoch': 3.0})

In [14]:
pred = trainer.predict(test_ds)

In [15]:
from sklearn.metrics import confusion_matrix 
y_pred = pred.predictions.argmax(axis=-1) 
y_true = pred.label_ids 
print(confusion_matrix(y_true, y_pred))

[[5172    4]
 [   7 4279]]


In [17]:
save_path = "saved_models/roberta_fake_news"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('saved_models/roberta_fake_news\\tokenizer_config.json',
 'saved_models/roberta_fake_news\\tokenizer.json')

In [18]:
import os 
print(os.getcwd())

C:\Users\pabit
